In [41]:
import pandas as pd
import time
from datetime import datetime
import os           

In [42]:
start=time.perf_counter()

# Load TRDL Data

In [43]:
from DatabaseCreator import DatabaseCreator
import CJDH_local_settings

#Run Database Creator
if __name__ == "__main__":
    db_creator = DatabaseCreator(db_settings=CJDH_local_settings.local_settings['FPL_Points_Predictor'])
    fpl_engine = db_creator.get_engine_for("fpl_data_analysis")

In [44]:
#add this info to a settings dict?
current_gw = 9
current_season = 20262027

In [45]:
#fpl_picks_history:
#combine fpl_picks and fpl_picks2 into a single sql table in the database
#add fpl_picks_{current_season} and fpl_picks_2_{current_season} into a single sql table in the database

In [46]:
# Erling Haaland goals vs xg in all seasons
view = db_creator.run_sql(f"""
SELECT player_name_id, season, event, fixture, goals_scored, expected_goals, expected_assists, assists
                          FROM playergw_transformed
                          WHERE expected_goals IS NOT NULL
                            AND player_name_id = 'Erling Haaland'
                          ORDER BY season, event, fixture
""")
view.pivot_table(index=['season'], values=['goals_scored', 'expected_goals', 'expected_assists', 'assists'], aggfunc='sum',margins=True,dropna=False)

,assists,expected_assists,expected_goals,goals_scored
season,,,,
20222023.0,9.0,1.33591,17.4943,36.0
20232024.0,8.0,2.18000,29.5700,27.0
20242025.0,3.0,2.04000,21.9000,22.0
20252026.0,8.0,2.67000,25.5000,27.0
20262027.0,0.0,0.85000,6.5600,9.0
All,28.0,9.07591,101.0243,121.0


In [47]:
# Erling Haaland goals vs xg in all seasons
view = db_creator.run_sql(f"""
SELECT *
FROM predictions
""")
view.columns

Index(['player_name_id', 'element', 'season', 'event', 'team_name', 'team_elo',
       'opp_team_name', 'opp_team_elo', 'team_attack_elo',
       'opp_team_attack_elo', 'team_defence_elo', 'opp_team_defence_elo',
       'position', 'value', 'fixture_count', 'mins_pred', 'xpoints',
       'xpoints_mins', 'xpoints_goals', 'xpoints_assists',
       'xg_overperf_factor', 'xa_overperf_factor', 'xpoints_clean_sheets',
       'xpoints_shots_saved', 'xpoints_goals_conceded', 'xpoints_defcon',
       'xpoints_bonus', 'xgp90_pred', 'xap90_pred', 'xgcp90_pred',
       'in_any_team', 'in_my_team', 'latest_any_team', 'latest_my_team',
       'prediction_timestamp', 'prediction_gameweek'],
      dtype='str')

In [48]:
print(f"\nOptimizing team selection for GW{current_gw+1} to GW38...")

from FPL_Optimizer import FPLOptimizer

remaining_gw_predictions = db_creator.run_sql(f"""
    
    WITH predictions_status AS (
    SELECT *
    FROM predictions
    LEFT JOIN player_status ON predictions.element = player_status.id
    )
    SELECT
        web_name,
        status,
        now_cost/10 as value,
        chance_of_playing_this_round,
        event,
        position,
        team_name,
        opp_team_name,
        xpoints,
        ROUND(mins_pred) as mins_pred
    FROM predictions_status
    WHERE latest_my_team = 1
        AND event BETWEEN {current_gw+1} AND 38
""")

if __name__ == "__main__":
    custom_columns = {
        'player_name': 'web_name',
        'team_name': 'team_name',
        'position': 'position',
        'value': 'value',
        'xpoints': 'xpoints'
    }
    
    optimizer = FPLOptimizer(
        df=remaining_gw_predictions[remaining_gw_predictions['event']==current_gw+1],
        budget=999.9,
        column_mapping=custom_columns
    )
    optimizer.optimize()

    # Get results as DataFrames
    # squad = optimizer.get_squad_dataframe()
    # starting_xi = optimizer.get_starting_xi_dataframe()
    result = optimizer.calculate_bench_boost_xpoints()
    result['event'] = current_gw+1



Optimizing team selection for GW10 to GW38...
FPL TEAM OPTIMIZER (Linear Programming)

Budget for 15-Player Squad: £999.9m

Solving optimization problem...

OPTIMAL 15-PLAYER SQUAD
Starting XI Formation: 4-4-2
Total Squad Cost: £100.0m / £999.9m
Total Weighted Expected Points: 54.04
Remaining Budget: £899.9m

--------------------------------GOALKEEPERS (2)---------------------------------
 Starter  Player Name                    Team         Cost     xPts 
--------------------------------------------------------------------------------
 🪑        Martinez                       Chelsea      £4.9     3.69 
 ✅        A.Becker                       Liverpool    £5.5     3.77 

---------------------------------DEFENDERS (5)----------------------------------
 Starter  Player Name                    Team         Cost     xPts 
--------------------------------------------------------------------------------
 🪑        Khalaili                       Crystal Palace £5.0     3.46 
 ✅        Justin

In [62]:
print(f"Team Selection Details - GW {current_gw+1}")
## latest_my_team and in_my_team differ because the former adjusts to freehits by setting the latest team to the previous gw
view = db_creator.run_sql(f"""
    
    WITH predictions_status AS (
    SELECT *
    FROM predictions
    LEFT JOIN player_status ON predictions.element = player_status.id
    )
    SELECT
        web_name,
        status,
        chance_of_playing_this_round,
        event,
        position,
        team_name,
        xpoints,
        CAST(ROUND(mins_pred) as INT) as mins_pred,
        CAST(ROUND(team_elo) as INT) as team_elo,
        CAST(ROUND(team_attack_elo) as INT) as team_attack_elo,
        CAST(ROUND(team_defence_elo) as INT) as team_defence_elo,
        opp_team_name,
        CAST(ROUND(opp_team_elo) as INT) as opp_team_elo,
        CAST(ROUND(opp_team_attack_elo) as INT) as opp_team_attack_elo,
        CAST(ROUND(opp_team_defence_elo) as INT) as opp_team_defence_elo,
        ROUND(xgp90_pred, 2) as xgp90_pred,
        ROUND(xg_overperf_factor, 2) as xg_overperf_factor,
        ROUND(xap90_pred, 2) as xap90_pred,
        ROUND(xa_overperf_factor, 2) as xa_overperf_factor,
        ROUND(xgcp90_pred, 2) as xgcp90_pred
        

    FROM predictions_status
    WHERE latest_my_team = 1
        AND event = {current_gw+1}
    ORDER BY xpoints DESC
""")
# print(f"Squad Points: {round(view['xpoints'].sum()+view['xpoints'][0],2)}") #including captain points
view#.head(15)

Team Selection Details - GW 10


,web_name,status,chance_of_playing_this_round,event,position,team_name,xpoints,mins_pred,team_elo,team_attack_elo,team_defence_elo,opp_team_name,opp_team_elo,opp_team_attack_elo,opp_team_defence_elo,xgp90_pred,xg_overperf_factor,xap90_pred,xa_overperf_factor,xgcp90_pred
0,Haaland,a,100.0,10.0,FWD,Man City,5.819661,86,1242,1317,1268,Nott'm Forest,1011,970,1154,0.66,1.17,0.22,2.81,1.14
1,Gibbs-White,a,100.0,10.0,MID,Nott'm Forest,5.079297,83,1011,970,1154,Man City,1242,1317,1268,0.28,1.00,0.29,1.86,1.80
2,Tavernier,a,NaN,10.0,MID,Bournemouth,5.023497,81,1108,1142,1182,Ipswich Town,1008,1059,959,0.27,1.00,0.29,1.28,1.18
3,Matheus N.,a,100.0,10.0,DEF,Man City,4.915788,78,1242,1317,1268,Nott'm Forest,1011,970,1154,0.05,1.00,0.18,2.48,0.93
4,Gakpo,a,100.0,10.0,MID,Liverpool,4.879842,68,1162,1165,1183,Crystal Palace,1002,941,960,0.40,1.00,0.28,1.35,1.10
5,Rayan,a,NaN,10.0,MID,Bournemouth,4.828528,84,1108,1142,1182,Ipswich Town,1008,1059,959,0.25,1.00,0.11,1.00,1.16
6,Thiago,a,NaN,10.0,FWD,Brentford,4.710636,86,1083,1130,1175,Brighton,1120,1250,1263,0.48,1.00,0.06,1.00,1.60
7,Calafiori,d,75.0,10.0,DEF,Arsenal,4.690791,60,1272,1197,1373,Hull City,1011,989,1075,0.08,1.00,0.15,1.68,0.86
8,Vuskovic,a,NaN,10.0,DEF,Brighton,4.476752,86,1120,1250,1263,Brentford,1083,1130,1175,0.07,1.00,0.06,1.00,1.24
9,Justin,a,NaN,10.0,DEF,Leeds,4.413936,83,1029,990,1179,Spurs,1005,980,1117,0.05,1.00,0.06,1.00,1.17


In [50]:
print(f"Best players to transfer out for the next 5 gameweeks: GW{current_gw+1} AND {current_gw+5}")
view = db_creator.run_sql(f"""
    SELECT
        player_name_id,
        latest_my_team,
        position,
        team_name,
        ROUND(AVG(opp_team_elo),1) AS avg_opp_team_elo,
        ROUND(SUM(xpoints),2) AS total_xpoints,
        ROUND(AVG(mins_pred),1) AS avg_mins_pred,
        ROUND(AVG(xgp90_pred),3) AS avg_xgp90_pred,
        ROUND(AVG(xg_overperf_factor),3) AS avg_xg_overperf_factor,
        ROUND(AVG(xap90_pred),3) AS avg_xap90_pred,
        ROUND(AVG(xa_overperf_factor),3) AS avg_xa_overperf_factor,
        ROUND(AVG(xgcp90_pred),3) AS avg_xgcp90_pred
    FROM predictions
    WHERE latest_my_team = 1
      AND event BETWEEN {current_gw+1} AND {current_gw+5}
      -- AND position = 'DEF'
      -- AND team_name IN ('Arsenal','Man United','Nottm Forest','Brentford','Brighton','Chelsea')
    GROUP BY player_name_id, position, team_name, latest_my_team
    ORDER BY total_xpoints ASC
""")
view.head(50)

Best players to transfer out for the next 5 gameweeks: GW10 AND 14


,player_name_id,latest_my_team,position,team_name,avg_opp_team_elo,total_xpoints,avg_mins_pred,avg_xgp90_pred,avg_xg_overperf_factor,avg_xap90_pred,avg_xa_overperf_factor,avg_xgcp90_pred
0,Emersonn Correia da Silva,1,FWD,Ipswich Town,1051.4,16.85,65.2,0.434,1.000,0.045,1.000,1.332
1,Emiliano Martínez Romero,1,GK,Chelsea,1043.5,18.18,89.4,NaN,1.000,NaN,1.000,1.422
2,Alisson Becker,1,GK,Liverpool,1049.2,18.34,90.0,NaN,1.000,NaN,1.000,1.295
3,Anan Khalaili,1,DEF,Crystal Palace,1075.9,18.51,75.6,0.059,1.000,0.081,1.000,1.530
4,Harvey Barnes,1,MID,Newcastle,1107.7,18.98,71.2,0.228,1.177,0.136,1.000,1.463
5,James Justin,1,DEF,Leeds,1067.8,20.05,83.2,0.050,1.000,0.057,1.000,1.387
6,Riccardo Calafiori,1,DEF,Arsenal,1080.0,21.66,61.4,0.076,1.000,0.127,1.683,1.068
7,Luka Vušković,1,DEF,Brighton,1054.4,22.57,85.9,0.066,1.000,0.065,1.000,1.264
8,Matheus Nunes,1,DEF,Man City,1086.5,22.70,77.9,0.046,1.000,0.175,2.479,1.127
9,Igor Thiago Nascimento Rodrigues,1,FWD,Brentford,1154.5,22.90,85.2,0.451,1.000,0.065,1.000,1.775


In [51]:
## Multi-week Optimizer

# from FPL_Optimizer import FPLOptimizer

# remaining_gw_predictions = db_creator.run_sql(f"""
    
#     WITH predictions_status AS (
#     SELECT *
#     FROM predictions
#     LEFT JOIN player_status ON predictions.element = player_status.id
#     )
#     SELECT
#         web_name,
#         status,
#         now_cost/10 as value,
#         chance_of_playing_this_round,
#         event,
#         position,
#         team_name,
#         opp_team_name,
#         xpoints,
#         ROUND(mins_pred) as mins_pred
#     FROM predictions_status
#     WHERE latest_my_team = 1
#         AND event BETWEEN {current_gw+1} AND 38
# """)

if __name__ == "__main__":
    custom_columns = {
        'player_name': 'web_name',
        'team_name': 'team_name',
        'position': 'position',
        'value': 'value',
        'xpoints': 'xpoints'
    }
    
    # 1. Initialize as an empty list to store DataFrames
    all_results_list = [] 

    for event in range(current_gw + 1, current_gw + 11):
        print(f"\nOptimizing team selection for GW{event}...")
        try:
            optimizer = FPLOptimizer(
                df=remaining_gw_predictions[remaining_gw_predictions['event'] == event],
                budget=102.4,
                column_mapping=custom_columns
            )
            optimizer.optimize()
        
            squad = optimizer.get_squad_dataframe()
            squad['event'] = event
            
            # 2. Append the individual DataFrame to the list
            all_results_list.append(squad) 

        except Exception as e:
            print(f"Error optimizing for GW{event}: {e}")

    # 3. Concatenate everything into one master DataFrame at the end
    if all_results_list:
        all_results = pd.concat(all_results_list, ignore_index=True)
        print("\nOptimization complete. Final DataFrame created.")
    else:
        all_results = pd.DataFrame() # Fallback if list is empty
        print("\nNo results found.")

all_results.pivot_table(index=['web_name','position'],columns='event',values='Starter',aggfunc='sum').reset_index().sort_values(by='position')


Optimizing team selection for GW10...
FPL TEAM OPTIMIZER (Linear Programming)

Budget for 15-Player Squad: £102.4m

Solving optimization problem...

OPTIMAL 15-PLAYER SQUAD
Starting XI Formation: 4-4-2
Total Squad Cost: £100.0m / £102.4m
Total Weighted Expected Points: 54.04
Remaining Budget: £2.4m

--------------------------------GOALKEEPERS (2)---------------------------------
 Starter  Player Name                    Team         Cost     xPts 
--------------------------------------------------------------------------------
 🪑        Martinez                       Chelsea      £4.9     3.69 
 ✅        A.Becker                       Liverpool    £5.5     3.77 

---------------------------------DEFENDERS (5)----------------------------------
 Starter  Player Name                    Team         Cost     xPts 
--------------------------------------------------------------------------------
 🪑        Khalaili                       Crystal Palace £5.0     3.46 
 ✅        Justin          

event,web_name,position,10,11,12,13,14,15,16,17,18,19
2,Calafiori,DEF,1,1,0,1,1,1,1,1,1,1
7,Justin,DEF,1,1,1,0,1,0,1,1,1,1
8,Khalaili,DEF,0,0,1,0,0,0,0,0,0,0
10,Matheus N.,DEF,1,1,1,1,1,1,1,1,1,1
14,Vuskovic,DEF,1,1,1,1,1,1,1,1,1,1
3,Emersonn,FWD,0,0,0,0,0,0,0,0,0,0
6,Haaland,FWD,1,1,1,1,1,1,1,1,1,1
13,Thiago,FWD,1,1,1,1,1,1,1,1,1,1
0,A.Becker,GK,1,0,0,1,1,1,1,1,0,1
9,Martinez,GK,0,1,1,0,0,0,0,0,1,0


In [52]:
print(f"Best players for next 5 gameweeks with exponential decay (normalized): GW{current_gw+1} to GW{current_gw+5}")

# For weights to sum to 1 with decay_factor d and initial weight w:
# w * (1 + d + d^2 + d^3 + d^4) = 1
# So w = 1 / (1 + d + d^2 + d^3 + d^4)

decay_factor = 0.5 #  lower decay factor means more emphasis on nearer gws, 1 is equal weights.
weights = [decay_factor**i for i in range(5)]
weight_sum = sum(weights)
initial_weight = 1 / weight_sum # weights add to 1
print(f"Decay factor: {decay_factor}, Weights: {[round((w/weight_sum),2) for w in weights]}")

view = db_creator.run_sql(f"""
    WITH 
    predictions_status AS (
        SELECT 
            p.player_name_id,
            p.element,
            ps.status,
            p.latest_my_team,
            p.position,
            p.team_name,
            p.opp_team_elo,
            p.xpoints,
            p.mins_pred,
            p.xgp90_pred,
            p.xg_overperf_factor,
            p.xap90_pred,
            p.xa_overperf_factor,
            p.xgcp90_pred,
            p.event,
            p.latest_any_team
        FROM predictions p
        LEFT JOIN player_status ps ON p.element = ps.id
    )
                                         
    SELECT
        player_name_id,
        status,
        latest_my_team,
        position,
        team_name,
        ROUND(SUM(
            CASE 
                WHEN event = {current_gw+1} THEN opp_team_elo * {initial_weight}
                WHEN event = {current_gw+2} THEN opp_team_elo * {initial_weight * decay_factor}
                WHEN event = {current_gw+3} THEN opp_team_elo * {initial_weight * decay_factor**2}
                WHEN event = {current_gw+4} THEN opp_team_elo * {initial_weight * decay_factor**3}
                WHEN event = {current_gw+5} THEN opp_team_elo * {initial_weight * decay_factor**4}
                ELSE 0
            END
        ), 1) AS avg_elo_decayed,
        ROUND(SUM(
            CASE 
                WHEN event = {current_gw+1} THEN xpoints * {initial_weight}
                WHEN event = {current_gw+2} THEN xpoints * {initial_weight * decay_factor}
                WHEN event = {current_gw+3} THEN xpoints * {initial_weight * decay_factor**2}
                WHEN event = {current_gw+4} THEN xpoints * {initial_weight * decay_factor**3}
                WHEN event = {current_gw+5} THEN xpoints * {initial_weight * decay_factor**4}
                ELSE 0
            END
        ), 2) AS total_xpoints_decayed,
        ROUND(AVG(mins_pred),1) AS avg_mins_pred,
        ROUND(AVG(xgp90_pred),3) AS avg_xgp90_pred,
        ROUND(AVG(xg_overperf_factor),3) AS avg_xg_overperf_factor,
        ROUND(AVG(xap90_pred),3) AS avg_xap90_pred,
        ROUND(AVG(xa_overperf_factor),3) AS avg_xa_overperf_factor,
        ROUND(AVG(xgcp90_pred),3) AS avg_xgcp90_pred
    FROM predictions_status
    WHERE (latest_any_team = 0 OR latest_my_team = 1)
      AND event BETWEEN {current_gw+1} AND {current_gw+5}
      AND status IN ('a','d')
      -- AND team_name IN ('Crystal Palace')
      -- AND position = 'FWD'
    GROUP BY player_name_id, status, position, team_name, latest_my_team
    ORDER BY total_xpoints_decayed DESC
""")
view.head(50)

Best players for next 5 gameweeks with exponential decay (normalized): GW10 to GW14
Decay factor: 0.5, Weights: [0.52, 0.26, 0.13, 0.06, 0.03]


,player_name_id,status,latest_my_team,position,team_name,avg_elo_decayed,total_xpoints_decayed,avg_mins_pred,avg_xgp90_pred,avg_xg_overperf_factor,avg_xap90_pred,avg_xa_overperf_factor,avg_xgcp90_pred
0,Erling Haaland,a,1,FWD,Man City,1055.0,5.74,84.9,0.636,1.172,0.215,2.809,1.202
1,Morgan Gibbs-White,a,1,MID,Nott'm Forest,1169.4,5.22,82.8,0.288,1.000,0.336,1.861,1.486
2,Phil Foden,a,0,MID,Man City,1055.0,5.03,61.2,0.592,1.602,0.188,1.000,0.992
3,Marcus Tavernier,a,1,MID,Bournemouth,1019.8,4.99,81.0,0.270,1.000,0.260,1.277,1.192
4,António João Pereira de Albuquerque Tavares da...,a,0,DEF,Bournemouth,1019.8,4.98,86.6,0.067,1.000,0.073,1.000,1.216
5,Cristhian Mosquera,a,0,DEF,Arsenal,1057.5,4.93,67.0,0.164,1.000,0.048,1.000,1.176
6,James Hill,a,0,DEF,Bournemouth,1019.8,4.92,85.8,0.038,1.000,0.052,1.000,1.222
7,Rayan Vitor Simplício Rocha,a,1,MID,Bournemouth,1019.8,4.82,84.2,0.253,1.000,0.105,1.000,1.184
8,Cody Gakpo,a,1,MID,Liverpool,1034.9,4.75,67.5,0.379,1.000,0.262,1.346,1.202
9,Matheus Nunes,a,1,DEF,Man City,1055.0,4.73,77.9,0.046,1.000,0.175,2.479,1.127


In [53]:
print(f"Best players to pick for the rest of the season: GW{current_gw+1} AND GW38")
view = db_creator.run_sql(f"""
WITH predictions_status AS (
    SELECT 
        p.player_name_id,
        p.element,
        ps.status,
        p.latest_my_team,
        p.position,
        p.team_name,
        p.opp_team_elo,
        p.xpoints,
        p.mins_pred,
        p.xgp90_pred,
        p.xg_overperf_factor,
        p.xap90_pred,
        p.xa_overperf_factor,
        p.xgcp90_pred,
        p.event,
        p.latest_any_team
    FROM predictions p
    LEFT JOIN player_status ps ON p.element = ps.id
)  
SELECT
    player_name_id,
    status,
    latest_my_team,
    position,
    team_name,
    ROUND(AVG(opp_team_elo),1) AS avg_opp_team_elo,
    ROUND(SUM(xpoints),2) AS total_xpoints,
    ROUND(AVG(mins_pred),1) AS avg_mins_pred,
    ROUND(AVG(xg_overperf_factor),3) AS avg_xg_overperf_factor,
    ROUND(AVG(xgp90_pred),3) AS avg_xgp90_pred,
    ROUND(AVG(xa_overperf_factor),3) AS avg_xa_overperf_factor,
    ROUND(AVG(xap90_pred),3) AS avg_xap90_pred,
    ROUND(AVG(xgcp90_pred),3) AS avg_xgcp90_pred
FROM predictions_status
WHERE (latest_any_team = 0 OR latest_my_team = 1)
    AND status IN ('a','d')
    -- AND position IN ('DEF')
    -- AND team_name = 'Liverpool'
  AND event BETWEEN {current_gw+1} AND 38
GROUP BY player_name_id, status, position, team_name, latest_my_team
ORDER BY total_xpoints DESC
""")
view.head(50)

Best players to pick for the rest of the season: GW10 AND GW38


,player_name_id,status,latest_my_team,position,team_name,avg_opp_team_elo,total_xpoints,avg_mins_pred,avg_xg_overperf_factor,avg_xgp90_pred,avg_xa_overperf_factor,avg_xap90_pred,avg_xgcp90_pred
0,Erling Haaland,a,1,FWD,Man City,1065.8,165.24,85.4,1.172,0.639,2.809,0.219,1.159
1,Morgan Gibbs-White,a,1,MID,Nott'm Forest,1079.9,156.76,82.8,1.000,0.293,1.861,0.356,1.419
2,Phil Foden,a,0,MID,Man City,1065.8,145.64,61.3,1.602,0.596,1.000,0.196,0.968
3,Marcus Tavernier,a,1,MID,Bournemouth,1068.3,143.16,81.1,1.000,0.272,1.277,0.255,1.261
4,Cristhian Mosquera,a,0,DEF,Arsenal,1065.0,142.66,67.1,1.000,0.164,1.000,0.048,1.102
5,Rayan Vitor Simplício Rocha,a,1,MID,Bournemouth,1068.3,137.87,84.2,1.000,0.255,1.000,0.103,1.268
6,António João Pereira de Albuquerque Tavares da...,a,0,DEF,Bournemouth,1068.3,136.96,86.6,1.000,0.067,1.000,0.073,1.299
7,James Hill,a,0,DEF,Bournemouth,1068.3,136.58,85.8,1.000,0.038,1.000,0.051,1.316
8,Alex Iwobi,a,0,MID,Fulham,1084.2,135.32,78.3,1.428,0.296,1.000,0.175,1.330
9,Igor Thiago Nascimento Rodrigues,a,1,FWD,Brentford,1079.0,134.55,85.9,1.000,0.456,1.000,0.067,1.544


In [54]:
# print(f"Best available players to pick for the next 5 gameweeks: GW{current_gw+1} AND {current_gw+5}")
# view = db_creator.run_sql(f"""
# WITH predictions_status AS (
#     SELECT 
#         p.player_name_id,
#         p.element,
#         ps.status,
#         p.latest_my_team,
#         p.position,
#         p.team_name,
#         p.opp_team_elo,
#         p.xpoints,
#         p.mins_pred,
#         p.xgp90_pred,
#         p.xg_overperf_factor,
#         p.xap90_pred,
#         p.xa_overperf_factor,
#         p.xgcp90_pred,
#         p.event,
#         p.latest_any_team
#     FROM predictions p
#     LEFT JOIN player_status ps ON p.element = ps.id
# )  
# SELECT
#         player_name_id,
#         status,
#         latest_my_team,
#         position,
#         team_name,
#         ROUND(AVG(opp_team_elo),1) AS avg_opp_team_elo,
#         ROUND(SUM(xpoints),2) AS total_xpoints,
#         ROUND(AVG(mins_pred),1) AS avg_mins_pred,
#         ROUND(AVG(xgp90_pred),3) AS avg_xgp90_pred,
#         ROUND(AVG(xg_overperf_factor),3) AS avg_xg_overperf_factor,
#         ROUND(AVG(xap90_pred),3) AS avg_xap90_pred,
#         ROUND(AVG(xa_overperf_factor),3) AS avg_xa_overperf_factor,
#         ROUND(AVG(xgcp90_pred),3) AS avg_xgcp90_pred
#     FROM predictions_status
#     WHERE (latest_any_team = 0 OR latest_my_team = 1)
#       AND event BETWEEN {current_gw+1} AND {current_gw+5}
#       AND status IN ('a','d')
#         --AND position = 'FWD'
#       -- AND team_name IN ('Arsenal','Man United','Nottm Forest','Brentford','Brighton','Chelsea')
#     GROUP BY player_name_id, status, position, team_name, latest_my_team
#     ORDER BY total_xpoints DESC
# """)
# view.head(50)

In [55]:
print(f"\nBest available players to replace worst in my team: {current_gw+1} to {current_gw+5}")

view = db_creator.run_sql(f"""
    WITH 
    predictions_status AS (
        SELECT 
            p.player_name_id,
            p.element,
            ps.status,
            p.latest_my_team,
            p.position,
            p.team_name,
            p.opp_team_elo,
            p.xpoints,
            p.mins_pred,
            p.xgp90_pred,
            p.xap90_pred,
            p.xgcp90_pred,
            p.event,
            p.latest_any_team
        FROM predictions p
        LEFT JOIN player_status ps ON p.element = ps.id
    ),

    summed AS (
        SELECT 
            player_name_id,
            team_name,
                          status,
            position,
            latest_my_team,
            latest_any_team,
            SUM(xpoints) AS total_xpoints
        FROM predictions_status
        WHERE event BETWEEN {current_gw+1} AND {current_gw+5}
          AND status IN ('a', 'd')
        GROUP BY player_name_id, team_name, status, position, latest_my_team, latest_any_team
    ),

    worst_gk AS (
        SELECT player_name_id, team_name, status, total_xpoints AS xpoints, latest_my_team, position
        FROM summed
        WHERE latest_my_team = 1
          AND position = 'GK'
        ORDER BY total_xpoints ASC
        LIMIT 1
    ),
    best_gk AS (
        SELECT player_name_id, team_name, status, total_xpoints AS xpoints, latest_my_team, position
        FROM summed
        WHERE latest_any_team = 0
          AND latest_my_team = 0
          AND position = 'GK'
    ),

    worst_def AS (
        SELECT player_name_id, team_name, status, total_xpoints AS xpoints, latest_my_team, position
        FROM summed
        WHERE latest_my_team = 1
          AND position = 'DEF'
        ORDER BY total_xpoints ASC
        LIMIT 1
    ),
    best_def AS (
        SELECT player_name_id, team_name, status, total_xpoints AS xpoints, latest_my_team, position
        FROM summed
        WHERE latest_any_team = 0
          AND latest_my_team = 0
          AND position = 'DEF'
    ),

    worst_mid AS (
        SELECT player_name_id, team_name, status, total_xpoints AS xpoints, latest_my_team, position
        FROM summed
        WHERE latest_my_team = 1
          AND position = 'MID'
        ORDER BY total_xpoints ASC
        LIMIT 1
    ),
    best_mid AS (
        SELECT player_name_id, team_name, status, total_xpoints AS xpoints, latest_my_team, position
        FROM summed
        WHERE latest_any_team = 0
          AND latest_my_team = 0
          AND position = 'MID'
    ),

    worst_fwd AS (
        SELECT player_name_id, team_name, status, total_xpoints AS xpoints, latest_my_team, position
        FROM summed
        WHERE latest_my_team = 1
          AND position = 'FWD'
        ORDER BY total_xpoints ASC
        LIMIT 1
    ),
    best_fwd AS (
        SELECT player_name_id, team_name, status, total_xpoints AS xpoints, latest_my_team, position
        FROM summed
        WHERE latest_any_team = 0
          AND latest_my_team = 0
          AND position = 'FWD'
    )

    SELECT 
        b.position,
        b.team_name AS best_team,
        b.status AS status, 
        b.player_name_id AS best_player_id,
        b.xpoints AS best_xpoints,
        w.player_name_id AS worst_player_id,
        w.xpoints AS worst_xpoints,
        (b.xpoints - w.xpoints) AS xpoints_diff
    FROM best_gk b CROSS JOIN worst_gk w

    UNION ALL

    SELECT 
        b.position,
        b.team_name AS best_team,
        b.status AS status, 
        b.player_name_id AS best_player_id,
        b.xpoints AS best_xpoints,
        w.player_name_id AS worst_player_id,
        w.xpoints AS worst_xpoints,
        (b.xpoints - w.xpoints) AS xpoints_diff
    FROM best_def b CROSS JOIN worst_def w

    UNION ALL

    SELECT 
        b.position,
        b.team_name AS best_team,
        b.status AS status, 
        b.player_name_id AS best_player_id,
        b.xpoints AS best_xpoints,
        w.player_name_id AS worst_player_id,
        w.xpoints AS worst_xpoints,
        (b.xpoints - w.xpoints) AS xpoints_diff
    FROM best_mid b CROSS JOIN worst_mid w

    UNION ALL

    SELECT 
        b.position,
        b.team_name AS best_team,
        b.status AS status,
        b.player_name_id AS best_player_id,
        b.xpoints AS best_xpoints,
        w.player_name_id AS worst_player_id,
        w.xpoints AS worst_xpoints,
        (b.xpoints - w.xpoints) AS xpoints_diff
    FROM best_fwd b CROSS JOIN worst_fwd w
    ORDER BY xpoints_diff DESC
""")
view.head(20)


Best available players to replace worst in my team: 10 to 14


,position,best_team,status,best_player_id,best_xpoints,worst_player_id,worst_xpoints,xpoints_diff
0,DEF,Bournemouth,a,António João Pereira de Albuquerque Tavares da...,24.458802,Anan Khalaili,18.514434,5.944368
1,DEF,Bournemouth,a,James Hill,24.436092,Anan Khalaili,18.514434,5.921659
2,MID,Man City,a,Phil Foden,24.858694,Harvey Barnes,18.979576,5.879118
3,DEF,Arsenal,a,Cristhian Mosquera,24.037890,Anan Khalaili,18.514434,5.523456
4,MID,Coventry City,a,Jack Rudoni,23.313022,Harvey Barnes,18.979576,4.333446
5,MID,Fulham,a,Alex Iwobi,23.200152,Harvey Barnes,18.979576,4.220576
6,MID,Spurs,a,Mohammed Kudus,23.024598,Harvey Barnes,18.979576,4.045022
7,DEF,Man City,a,Abdukodir Khusanov,21.662740,Anan Khalaili,18.514434,3.148307
8,MID,Brighton,a,Yasin Ayari,22.080331,Harvey Barnes,18.979576,3.100755
9,DEF,Bournemouth,a,Adrien Truffert,21.450820,Anan Khalaili,18.514434,2.936386


In [56]:
player_name = 'Erling Haaland'
print(f"\n {player_name} xPoints between {current_gw+1} and {current_gw+5}:")
view = db_creator.run_sql(f"""
    SELECT
        player_name_id,
        event,
        position,
        team_name,
        opp_team_name,
        value/10 as value,
        opp_team_elo,
        xpoints,
        mins_pred,
        xgp90_pred,
        xap90_pred,
        xgcp90_pred
    FROM predictions
    WHERE event BETWEEN {current_gw+1} AND {current_gw+5}
        AND player_name_id = '{player_name}'
        -- AND value <= 48
    ORDER BY event ASC
""")
view.head(20)                                                              



 Erling Haaland xPoints between 10 and 14:


,player_name_id,event,position,team_name,opp_team_name,value,opp_team_elo,xpoints,mins_pred,xgp90_pred,xap90_pred,xgcp90_pred
0,Erling Haaland,10.0,FWD,Man City,Nott'm Forest,15.8,1011.326815,5.819661,85.596932,0.663159,0.218967,1.143529
1,Erling Haaland,11.0,FWD,Man City,Fulham,15.8,1036.650017,5.785508,85.711633,0.658420,0.215693,1.162017
2,Erling Haaland,12.0,FWD,Man City,Arsenal,15.8,1272.232293,5.345174,81.919741,0.596481,0.208215,1.435892
3,Erling Haaland,13.0,FWD,Man City,Leeds,15.8,1028.829679,5.692938,85.742324,0.635572,0.216572,1.182118
4,Erling Haaland,14.0,FWD,Man City,Brentford,15.8,1083.424513,5.620072,85.711633,0.624110,0.213477,1.086843


# Chip Decisions

### Triple Captain

In [57]:
print("When to use TRIPLE CAPTAIN chip (gws1-18)")
view = db_creator.run_sql(f"""
    SELECT
        player_name_id,
        event,
        position,
        team_name,
        opp_team_name,
        xpoints
    FROM predictions
    WHERE in_my_team = 1
      AND event BETWEEN {current_gw} AND 18
    ORDER BY xpoints DESC
""")
view.head(10)

When to use TRIPLE CAPTAIN chip (gws1-18)


,player_name_id,event,position,team_name,opp_team_name,xpoints
0,Erling Haaland,16.0,FWD,Man City,Hull City,5.825770
1,Erling Haaland,10.0,FWD,Man City,Nott'm Forest,5.819661
2,Erling Haaland,11.0,FWD,Man City,Fulham,5.785508
3,Erling Haaland,18.0,FWD,Man City,Everton,5.743892
4,Erling Haaland,13.0,FWD,Man City,Leeds,5.692938
5,Morgan Gibbs-White,15.0,MID,Nott'm Forest,Sunderland,5.624216
6,Erling Haaland,17.0,FWD,Man City,Newcastle,5.624201
7,Erling Haaland,14.0,FWD,Man City,Brentford,5.620072
8,Erling Haaland,15.0,FWD,Man City,Chelsea,5.604576
9,Morgan Gibbs-White,13.0,MID,Nott'm Forest,Hull City,5.523236


### Bench Boost

In [58]:
print(f"\nOptimizing team selection for GW{current_gw+1} to GW38...")

if __name__ == "__main__":
    custom_columns = {
        'player_name': 'web_name',
        'team_name': 'team_name',
        'position': 'position',
        'value': 'value',
        'xpoints': 'xpoints'
    }
    all_results =[]
    for event in range(current_gw+1, 39):
        print(f"\nOptimizing team selection for GW{event}...")
        try:
            optimizer = FPLOptimizer(
                df=remaining_gw_predictions[remaining_gw_predictions['event']==event],
                budget=999.9,
                column_mapping=custom_columns
            )
            optimizer.optimize()
        
            # Get results as DataFrames
            # squad = optimizer.get_squad_dataframe()
            # starting_xi = optimizer.get_starting_xi_dataframe()
            result = optimizer.calculate_bench_boost_xpoints()
            result['event'] = event
            # Append just the relevant keys as a dict
            all_results.append({
                'event': result['event'],
                'bench_boost_gain': result['bench_boost_gain']
            })
        except Exception as e:
            print(f"Error optimizing for GW{event}: {e}")



Optimizing team selection for GW10 to GW38...

Optimizing team selection for GW10...
FPL TEAM OPTIMIZER (Linear Programming)

Budget for 15-Player Squad: £999.9m

Solving optimization problem...

OPTIMAL 15-PLAYER SQUAD
Starting XI Formation: 4-4-2
Total Squad Cost: £100.0m / £999.9m
Total Weighted Expected Points: 54.04
Remaining Budget: £899.9m

--------------------------------GOALKEEPERS (2)---------------------------------
 Starter  Player Name                    Team         Cost     xPts 
--------------------------------------------------------------------------------
 🪑        Martinez                       Chelsea      £4.9     3.69 
 ✅        A.Becker                       Liverpool    £5.5     3.77 

---------------------------------DEFENDERS (5)----------------------------------
 Starter  Player Name                    Team         Cost     xPts 
--------------------------------------------------------------------------------
 🪑        Khalaili                       Crystal


OPTIMAL 15-PLAYER SQUAD
Starting XI Formation: 4-4-2
Total Squad Cost: £100.0m / £999.9m
Total Weighted Expected Points: 51.54
Remaining Budget: £899.9m

--------------------------------GOALKEEPERS (2)---------------------------------
 Starter  Player Name                    Team         Cost     xPts 
--------------------------------------------------------------------------------
 🪑        A.Becker                       Liverpool    £5.5     3.66 
 ✅        Martinez                       Chelsea      £4.9     3.69 

---------------------------------DEFENDERS (5)----------------------------------
 Starter  Player Name                    Team         Cost     xPts 
--------------------------------------------------------------------------------
 🪑        Calafiori                      Arsenal      £5.9     3.54 
 ✅        Khalaili                       Crystal Palace £5.0     3.92 
 ✅        Justin                         Leeds        £4.5     4.40 
 ✅        Vuskovic                 

In [59]:
print("Best weeks to Bench Boost:")
all_results_df = pd.DataFrame(all_results).sort_values(by='bench_boost_gain', ascending=False, inplace=False)
all_results_df

Best weeks to Bench Boost:


,event,bench_boost_gain
20,30,13.274625
24,34,13.082072
4,14,13.082072
6,16,13.068547
17,27,13.068547
21,31,13.060773
27,37,13.023766
14,24,12.961200
8,18,12.937961
1,11,12.919517


In [60]:
finish=time.perf_counter()
print(f'Finished in {round(((finish-start)/60),2)} minute(s)')
# Finished in 0.01 minute(s)

Finished in 0.1 minute(s)
